In [11]:
from pathlib import Path
import pandas as pd
from ucimlrepo import fetch_ucirepo

pd.set_option("display.float_format", "{:.2f}".format)

# El notebook corre desde notebooks/, así que subimos un nivel
RAIZ = Path.cwd().parent
RAW = RAIZ / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
print("Datos se guardarán en:", RAW)

Datos se guardarán en: c:\Users\danie\Documents\predictor-concreto\data\raw


In [12]:
dataset = fetch_ucirepo(id=165)
X = dataset.data.features
y = dataset.data.targets

df_crudo = pd.concat([X, y], axis=1)
df_crudo.to_csv(RAW / "concrete_uci_original.csv", index=False)

print("Forma:", df_crudo.shape)
print(df_crudo.columns.tolist())

Forma: (1030, 9)
['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age', 'Concrete compressive strength']


In [13]:
df = df_crudo.copy()
df.columns = ["cemento", "escoria", "ceniza", "agua", "superplast",
              "agr_grueso", "agr_fino", "edad", "resistencia"]

print("Forma:", df.shape)
df.head()

Forma: (1030, 9)


,cemento,escoria,ceniza,agua,superplast,agr_grueso,agr_fino,edad,resistencia
0,540.00,0.00,0.00,162.00,2.50,1040.00,676.00,28,79.99
1,540.00,0.00,0.00,162.00,2.50,1055.00,676.00,28,61.89
2,332.50,142.50,0.00,228.00,0.00,932.00,594.00,270,40.27
3,332.50,142.50,0.00,228.00,0.00,932.00,594.00,365,41.05
4,198.60,132.40,0.00,192.00,0.00,978.40,825.50,360,44.30


In [14]:
print("Tipos de dato:\n", df.dtypes, "\n")
print("Valores faltantes:\n", df.isna().sum(), "\n")
print("Filas duplicadas:", df.duplicated().sum())

Tipos de dato:
 cemento        float64
escoria        float64
ceniza         float64
agua           float64
superplast     float64
agr_grueso     float64
agr_fino       float64
edad             int64
resistencia    float64
dtype: object 

Valores faltantes:
 cemento        0
escoria        0
ceniza         0
agua           0
superplast     0
agr_grueso     0
agr_fino       0
edad           0
resistencia    0
dtype: int64 

Filas duplicadas: 25


In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
cemento,1030.00,281.17,104.51,102.00,192.38,272.90,350.00,540.00
escoria,1030.00,73.90,86.28,0.00,0.00,22.00,142.95,359.40
ceniza,1030.00,54.19,64.00,0.00,0.00,0.00,118.30,200.10
agua,1030.00,181.57,21.35,121.80,164.90,185.00,192.00,247.00
superplast,1030.00,6.20,5.97,0.00,0.00,6.40,10.20,32.20
agr_grueso,1030.00,972.92,77.75,801.00,932.00,968.00,1029.40,1145.00
agr_fino,1030.00,773.58,80.18,594.00,730.95,779.50,824.00,992.60
edad,1030.00,45.66,63.17,1.00,7.00,28.00,56.00,365.00
resistencia,1030.00,35.82,16.71,2.33,23.71,34.45,46.14,82.60


In [16]:
dup = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print("Filas involucradas (originales + copias):", len(dup))
dup.head(10)

Filas involucradas (originales + copias): 36


,cemento,escoria,ceniza,agua,superplast,agr_grueso,agr_fino,edad,resistencia
801,252.00,0.00,0.00,185.00,0.00,1111.00,784.00,28,19.69
809,252.00,0.00,0.00,185.00,0.00,1111.00,784.00,28,19.69
83,362.60,189.00,0.00,164.90,11.60,944.70,755.80,3,35.30
86,362.60,189.00,0.00,164.90,11.60,944.70,755.80,3,35.30
88,362.60,189.00,0.00,164.90,11.60,944.70,755.80,3,35.30
91,362.60,189.00,0.00,164.90,11.60,944.70,755.80,3,35.30
106,362.60,189.00,0.00,164.90,11.60,944.70,755.80,7,55.90
109,362.60,189.00,0.00,164.90,11.60,944.70,755.80,7,55.90
111,362.60,189.00,0.00,164.90,11.60,944.70,755.80,7,55.90
129,362.60,189.00,0.00,164.90,11.60,944.70,755.80,28,71.30


In [17]:
df["edad"].value_counts().sort_index()

edad
1        2
3      134
7      126
14      62
28     425
56      91
90      54
91      22
100     52
120      3
180     26
270     13
360      6
365     14
Name: count, dtype: int64

In [18]:
masa = df[["cemento", "escoria", "ceniza", "agua", "superplast",
           "agr_grueso", "agr_fino"]].sum(axis=1)
masa.describe()

count   1030.00
mean    2343.52
std       65.37
min     2194.60
25%     2291.15
50%     2349.10
75%     2390.40
max     2551.00
dtype: float64

In [19]:
df_limpio = df.drop_duplicates().reset_index(drop=True)
print("Antes:", df.shape, "| Después:", df_limpio.shape)

PROC = RAIZ / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
df_limpio.to_csv(PROC / "concreto_limpio.csv", index=False)

Antes: (1030, 9) | Después: (1005, 9)


In [20]:
comp = ["cemento", "escoria", "ceniza", "agua",
        "superplast", "agr_grueso", "agr_fino"]

n_mezclas = df_limpio.groupby(comp).ngroups
print("Mezclas distintas:", n_mezclas, "de", len(df_limpio), "filas")

df_limpio.groupby(comp).size().describe()

Mezclas distintas: 427 de 1005 filas


count   427.00
mean      2.35
std       1.86
min       1.00
25%       1.00
50%       1.00
75%       4.00
max      12.00
dtype: float64

## Conclusiones de la auditoría

- 1030 filas y 9 columnas, sin valores faltantes; todas las variables son numéricas.
- 25 filas duplicadas exactas (cargas repetidas): se eliminan. Dataset limpio: 1005 filas.
- Resistencia entre 2.3 y 82.6 MPa: concreto convencional y de alta resistencia, no UHPC.
- Escoria, ceniza y superplastificante tienen ceros reales (mezclas sin ese componente).
- La masa total por m³ es de 2195 a 2551 kg/m³ (media 2343): coherente con concreto fresco.
- Una misma mezcla aparece a varias edades: la validación debe agrupar por mezcla.
- Edad concentrada en 28 días (cerca del 41%); pocos datos a edades extremas.